# 01 — Transform Customers to Silver

## Purpose

Transform validated customer events from Bronze into a clean, current-state Silver customer table.

This notebook will:

- Read customer events from the Bronze Delta table
- Standardize data types and business values
- Deduplicate source events
- Apply CDC operations using the latest event per customer
- Exclude customers whose latest operation is `DELETE`
- Validate the resulting customer records
- Persist the final dataset as a managed Delta table

## Source

- `workspace.revenue_leakage_bronze.customer_events`

## Target

- `workspace.revenue_leakage_silver.customers`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_CUSTOMERS_TABLE = (
    "workspace.revenue_leakage_bronze.customer_events"
)

SILVER_CUSTOMERS_TABLE = (
    "workspace.revenue_leakage_silver.customers"
)

bronze_customer_events_df = spark.table(
    BRONZE_CUSTOMERS_TABLE
)

print(
    f"Bronze customer events: "
    f"{bronze_customer_events_df.count():,}"
)

bronze_customer_events_df.printSchema()

display(
    bronze_customer_events_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

## 1. Standardize and Resolve Customer CDC Events

Clean the Bronze customer attributes, remove exact duplicate events, and rank each customer's history by event time.

For every `customer_id`, the most recent event represents the current state:

- `INSERT` creates a customer
- `UPDATE` replaces the previous customer state
- `DELETE` removes the customer from the active Silver dataset

In [0]:
customer_cdc_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("event_timestamp").desc(),
        F.col("_source_file_modification_time").desc(),
        F.col("_ingested_at").desc(),
        F.col("_record_hash").desc(),
    )
)

standardized_customer_events_df = (
    bronze_customer_events_df
    .dropDuplicates(["_record_hash"])
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id")),
    )
    .withColumn(
        "first_name",
        F.initcap(F.trim(F.col("first_name"))),
    )
    .withColumn(
        "last_name",
        F.initcap(F.trim(F.col("last_name"))),
    )
    .withColumn(
        "email",
        F.lower(F.trim(F.col("email"))),
    )
    .withColumn(
        "country",
        F.trim(F.col("country")),
    )
    .withColumn(
        "region",
        F.trim(F.col("region")),
    )
    .withColumn(
        "customer_segment",
        F.trim(F.col("customer_segment")),
    )
    .withColumn(
        "customer_status",
        F.initcap(F.trim(F.col("customer_status"))),
    )
    .withColumn(
        "operation",
        F.upper(F.trim(F.col("operation"))),
    )
)

latest_customer_events_df = (
    standardized_customer_events_df
    .withColumn(
        "_cdc_rank",
        F.row_number().over(customer_cdc_window),
    )
    .filter(F.col("_cdc_rank") == 1)
    .drop("_cdc_rank")
)

silver_customers_df = (
    latest_customer_events_df
    .filter(F.col("operation") != "DELETE")
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "region",
        "customer_segment",
        "signup_date",
        "customer_status",
        F.col("operation").alias("last_operation"),
        F.col("event_timestamp").alias(
            "last_event_timestamp"
        ),
        "_source_system",
        "_source_entity",
        "_source_file_path",
        "_record_hash",
        F.current_timestamp().alias(
            "_silver_processed_at"
        ),
    )
)

print(
    f"Bronze events: "
    f"{bronze_customer_events_df.count():,}"
)

print(
    f"Distinct Bronze events: "
    f"{standardized_customer_events_df.count():,}"
)

print(
    f"Latest customer states: "
    f"{latest_customer_events_df.count():,}"
)

print(
    f"Active Silver customers: "
    f"{silver_customers_df.count():,}"
)

display(
    latest_customer_events_df
    .groupBy("operation")
    .count()
    .orderBy("operation")
)

## 2. Validate the Current Silver Customer State

Validate the transformed customer dataset before persistence.

The checks cover:

- Required business fields
- Unique customer identifiers
- Valid email structure
- Valid signup dates
- Customer status and segment domains
- Exclusion of deleted customers
- Expected current-state row count

In [0]:
customer_status_values = sorted(
    row["customer_status"]
    for row in (
        silver_customers_df
        .select("customer_status")
        .distinct()
        .collect()
    )
)

customer_segment_values = sorted(
    row["customer_segment"]
    for row in (
        silver_customers_df
        .select("customer_segment")
        .distinct()
        .collect()
    )
)

country_values = sorted(
    row["country"]
    for row in (
        silver_customers_df
        .select("country")
        .distinct()
        .collect()
    )
)

print(
    f"Customer statuses: "
    f"{customer_status_values}"
)

print(
    f"Customer segments: "
    f"{customer_segment_values}"
)

print(
    f"Countries: "
    f"{country_values}"
)

display(
    silver_customers_df
    .groupBy(
        "customer_status",
        "customer_segment",
    )
    .count()
    .orderBy(
        "customer_status",
        "customer_segment",
    )
)

In [0]:
EXPECTED_ACTIVE_CUSTOMER_COUNT = 5_150

ALLOWED_CUSTOMER_STATUSES = [
    "Active",
    "Inactive",
]

ALLOWED_CUSTOMER_SEGMENTS = [
    "Premium",
    "Standard",
    "VIP",
]

ALLOWED_COUNTRIES = [
    "Canada",
    "France",
    "Germany",
    "Netherlands",
    "United Kingdom",
    "United States",
]

required_customer_columns = [
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "country",
    "region",
    "customer_segment",
    "signup_date",
    "customer_status",
    "last_event_timestamp",
]

required_field_is_missing = None

for column_name in required_customer_columns:
    missing_condition = (
        F.col(column_name).isNull()
        | (
            F.trim(
                F.col(column_name).cast("string")
            )
            == ""
        )
    )

    required_field_is_missing = (
        missing_condition
        if required_field_is_missing is None
        else required_field_is_missing
        | missing_condition
    )

invalid_email_condition = ~F.col("email").rlike(
    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
)

invalid_signup_date_condition = (
    F.col("signup_date")
    > F.to_date(F.col("last_event_timestamp"))
)

invalid_status_condition = (
    ~F.col("customer_status").isin(
        ALLOWED_CUSTOMER_STATUSES
    )
)

invalid_segment_condition = (
    ~F.col("customer_segment").isin(
        ALLOWED_CUSTOMER_SEGMENTS
    )
)

invalid_country_condition = (
    ~F.col("country").isin(ALLOWED_COUNTRIES)
)

invalid_operation_condition = (
    ~F.col("last_operation").isin(
        "INSERT",
        "UPDATE",
    )
)

customer_validation_metrics = (
    silver_customers_df
    .agg(
        F.count("*").alias(
            "silver_customer_count"
        ),
        F.countDistinct("customer_id").alias(
            "distinct_customer_count"
        ),
        F.sum(
            F.when(
                required_field_is_missing,
                1,
            ).otherwise(0)
        ).alias("null_required_field_count"),
        F.sum(
            F.when(
                invalid_email_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_email_count"),
        F.sum(
            F.when(
                invalid_signup_date_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_signup_date_count"),
        F.sum(
            F.when(
                invalid_status_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_status_count"),
        F.sum(
            F.when(
                invalid_segment_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_segment_count"),
        F.sum(
            F.when(
                invalid_country_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_country_count"),
        F.sum(
            F.when(
                invalid_operation_condition,
                1,
            ).otherwise(0)
        ).alias("invalid_operation_count"),
    )
    .first()
    .asDict()
)

duplicate_customer_count = (
    customer_validation_metrics[
        "silver_customer_count"
    ]
    - customer_validation_metrics[
        "distinct_customer_count"
    ]
)

for metric_name, metric_value in (
    customer_validation_metrics.items()
):
    print(
        f"{metric_name}: "
        f"{metric_value:,}"
    )

print(
    f"duplicate_customer_count: "
    f"{duplicate_customer_count:,}"
)

assert (
    customer_validation_metrics[
        "silver_customer_count"
    ]
    == EXPECTED_ACTIVE_CUSTOMER_COUNT
), "Unexpected Silver customer count."

assert duplicate_customer_count == 0, (
    "Duplicate customer IDs detected."
)

for metric_name in [
    "null_required_field_count",
    "invalid_email_count",
    "invalid_signup_date_count",
    "invalid_status_count",
    "invalid_segment_count",
    "invalid_country_count",
    "invalid_operation_count",
]:
    assert (
        customer_validation_metrics[metric_name]
        == 0
    ), f"Validation failed: {metric_name}"

print(
    "Silver customer validation completed successfully."
)

## 3. Persist the Silver Customer Table

Persist the validated current-state dataset as a managed Delta table.

The write process uses `customer_id` as the business key:

- Existing customers are updated
- New customers are inserted
- Customers absent from the resolved active state are deleted
- Reprocessing produces the same final customer population

In [0]:
spark.sql(
    """
    CREATE SCHEMA IF NOT EXISTS
    workspace.revenue_leakage_silver
    """
)

silver_customers_df.createOrReplaceTempView(
    "silver_customer_updates"
)

if spark.catalog.tableExists(
    SILVER_CUSTOMERS_TABLE
):
    spark.sql(
        f"""
        MERGE INTO {SILVER_CUSTOMERS_TABLE} AS target
        USING silver_customer_updates AS source
          ON target.customer_id = source.customer_id

        WHEN MATCHED
            AND target._record_hash <> source._record_hash
            THEN
        UPDATE SET *

        WHEN NOT MATCHED THEN
          INSERT *

        WHEN NOT MATCHED BY SOURCE THEN
          DELETE
        """
    )

    write_method = "Delta MERGE"

else:
    (
        silver_customers_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_CUSTOMERS_TABLE)
    )

    write_method = "Initial Delta table creation"

saved_silver_customers_df = spark.table(
    SILVER_CUSTOMERS_TABLE
)

saved_silver_customer_count = (
    saved_silver_customers_df.count()
)

saved_distinct_customer_count = (
    saved_silver_customers_df
    .select("customer_id")
    .distinct()
    .count()
)

assert (
    saved_silver_customer_count
    == EXPECTED_ACTIVE_CUSTOMER_COUNT
), "Saved Silver row count is incorrect."

assert (
    saved_distinct_customer_count
    == saved_silver_customer_count
), "Saved Silver table contains duplicate customers."

print(
    f"Write method: {write_method}"
)

print(
    f"Silver table: "
    f"{SILVER_CUSTOMERS_TABLE}"
)

print(
    f"Saved Silver customers: "
    f"{saved_silver_customer_count:,}"
)

print(
    f"Saved distinct customer IDs: "
    f"{saved_distinct_customer_count:,}"
)

display(
    saved_silver_customers_df
    .groupBy(
        "customer_status",
        "customer_segment",
    )
    .count()
    .orderBy(
        "customer_status",
        "customer_segment",
    )
)

## 4. Validate Idempotent Reprocessing

Reapply the same resolved customer state through the Delta merge and confirm that no duplicate customers or additional rows are created.

A successful rerun must preserve:

- The same total customer count
- One row per `customer_id`
- The same active customer population

In [0]:
latest_silver_history_df = (
    spark.sql(
        f"""
        DESCRIBE HISTORY
        {SILVER_CUSTOMERS_TABLE}
        LIMIT 1
        """
    )
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics",
    )
)

latest_history_row = (
    latest_silver_history_df.first()
)

operation_metrics = (
    latest_history_row["operationMetrics"]
    or {}
)

rows_inserted = int(
    operation_metrics.get(
        "numTargetRowsInserted",
        "0",
    )
)

rows_updated = int(
    operation_metrics.get(
        "numTargetRowsUpdated",
        "0",
    )
)

rows_deleted = int(
    operation_metrics.get(
        "numTargetRowsDeleted",
        "0",
    )
)

current_silver_customers_df = spark.table(
    SILVER_CUSTOMERS_TABLE
)

current_row_count = (
    current_silver_customers_df.count()
)

current_distinct_customer_count = (
    current_silver_customers_df
    .select("customer_id")
    .distinct()
    .count()
)

duplicate_customer_count = (
    current_row_count
    - current_distinct_customer_count
)

assert (
    latest_history_row["operation"]
    == "MERGE"
), "The latest Delta operation was not MERGE."

assert rows_inserted == 0, (
    "Idempotency failed: rows were inserted."
)

assert rows_updated == 0, (
    "Idempotency failed: unchanged rows were updated."
)

assert rows_deleted == 0, (
    "Idempotency failed: rows were deleted."
)

assert (
    current_row_count
    == EXPECTED_ACTIVE_CUSTOMER_COUNT
), "Unexpected customer count after rerun."

assert duplicate_customer_count == 0, (
    "Duplicate customers detected after rerun."
)

print(
    f"Rows inserted during rerun: "
    f"{rows_inserted:,}"
)

print(
    f"Rows updated during rerun: "
    f"{rows_updated:,}"
)

print(
    f"Rows deleted during rerun: "
    f"{rows_deleted:,}"
)

print(
    f"Customers after rerun: "
    f"{current_row_count:,}"
)

print(
    f"Duplicate customers after rerun: "
    f"{duplicate_customer_count:,}"
)

print(
    "Silver customer transformation is idempotent."
)

display(latest_silver_history_df)

## Result

The customer Silver transformation completed successfully:

- 5,500 Bronze customer events processed
- 5,200 latest customer states resolved
- 50 deleted customers excluded
- 5,150 active customer records persisted
- Zero duplicate customer identifiers
- Zero data-quality violations
- Delta merge rerun produced zero changes
- Target confirmed as a managed Delta table